# Christina Doty

In [ ]:
import pandas as pd
import numpy as np
import os
import pathlib as Path

# Tidy Crop Data

In [ ]:
def tidy_crop(read_path, write_path, crop='wheat', granularity='state'):
    if granularity == 'state':
        keep_cols = ["year", "state", "state_abbr", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "commodity"]
    elif granularity == 'county':
        keep_cols = ["year", "state", "state_abbr", "county", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "county", "commodity"]
    else:
        print("granularity should be either state or county")
        return None
    
    if crop == 'corn':
        keep_cols.append("util_practice_desc")

    df = pd.read_csv(read_path)
    # remove the duplicated yield rows that have dollar amounts instead of BU
    df_no_money = df.loc[df['unit'] != '$']
    df_usefulcols = df_no_money[keep_cols]

    if crop == 'wheat':
        df_pivot = df_usefulcols.pivot_table(index=pivot_index_cols, columns="statistic_category", values="value", aggfunc="first")
        df_pivot.columns.name=None
        df_tidy = df_pivot.reset_index()
        print(f"Sanity check, this should be 4: {len(df_usefulcols) / len(df_tidy)}")
    elif crop == 'corn':
        # Split into planted and harvest groups
        df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
        df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

        # For harvest rows, combine the stat category and util practice into one label
        # # e.g. "area harvested grain", "production silage", "yield grain" etc.
        df_harvest = df_harvest.copy()
        df_harvest["stat_label"] = (df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower())
        
        # remove the extra area planted entries that appear for the grain category
        df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

        # For planted rows, the label is just the stat category
        df_planted = df_planted.copy()
        df_planted["stat_label"] = df_planted["statistic_category"]

        # Combine and pivot once on the new label
        df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

        df_tidy = df_combined.pivot_table(index=pivot_index_cols, columns="stat_label", values="value", aggfunc="first").reset_index()
        df_tidy.columns.name = None
    else:
        print("crop should be either wheat or corn")
        return None
    
    df_tidy.to_csv(write_path)
    return df_tidy

In [ ]:
wheat_state_tidy = tidy_crop("../data/wheat_state.csv", "../data/wheat_state_tidy.csv", crop='wheat', granularity='state')
print(len(wheat_state_tidy))
wheat_state_tidy.head(5)

In [ ]:
wheat_county_tidy = tidy_crop("../data/wheat_county.csv", "../data/wheat_county_tidy.csv", crop='wheat', granularity='county')
print(len(wheat_county_tidy))
wheat_county_tidy.head(5)

In [ ]:
corn_state_tidy = tidy_crop("../data/corn_state.csv", "../data/corn_state_tidy.csv", crop='corn', granularity='state')
print(len(corn_state_tidy))
corn_state_tidy.head(5)

In [ ]:
corn_county_tidy = tidy_crop("../data/corn_county.csv", "../data/corn_county_tidy.csv", crop='corn', granularity='county')
print(len(corn_county_tidy))
corn_county_tidy.head(5)

# Scratch Work Below

In [ ]:
raw_df = pd.read_csv("../data/corn_state.csv")
raw_df.head(2)

In [ ]:
df_no_money = raw_df.loc[raw_df['unit'] != '$']

In [ ]:
df_usefulcols = df_no_money[["year", "state", "state_abbr", "commodity", "util_practice_desc", "statistic_category", "value"]]
df_usefulcols.head(2)

In [ ]:
df_pivot = df_usefulcols.pivot_table(index=["year", "state", "state_abbr", "util_practice_desc", "commodity"], columns="statistic_category", values="value", aggfunc="first")
df_pivot.columns.name=None
df_pivot = df_pivot.reset_index()
df_pivot.head(5)

In [ ]:
############ Doesn't work

# Split into the three groups
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# Pivot each separately
planted_pivot = df_planted.pivot_table(index=["year", "state", "state_abbr", "commodity"], columns="statistic_category", values="value", aggfunc="first").reset_index()
planted_pivot.columns.name = None

# Will only have 'area planted', rename to be explicit
#planted_pivot = planted_pivot.rename(columns={"area planted": "area planted"})

harvest_pivot = df_harvest.pivot_table(
    index=["year", "state", "state_abbr", "commodity", "util_practice_desc"],
    columns="statistic_category",
    values="value",
    aggfunc="first"
).reset_index()
harvest_pivot.columns.name = None

# Merge area planted onto each grain/silage row
df_final = harvest_pivot.merge(
    planted_pivot[["year", "state", "state_abbr", "commodity", "AREA PLANTED"]],
    on=["year", "state", "state_abbr", "commodity"],
    how="left"
)
df_final.head(10)

In [ ]:
# Split into planted and harvest groups as before
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# For harvest rows, combine the stat category and util practice into one label
df_harvest = df_harvest.copy()
df_harvest["stat_label"] = (
    df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower()
)
# e.g. "area harvested grain", "production silage", "yield grain" etc.

df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

# For planted rows, the label is just the stat category + total
df_planted = df_planted.copy()
df_planted["stat_label"] = df_planted["statistic_category"] + " total"
# e.g. "area planted total"

# Combine and pivot once on the new label
df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

df_final = df_combined.pivot_table(
    index=["year", "state"],
    columns="stat_label",
    values="value",
    aggfunc="first"
).reset_index()
df_final.columns.name = None

In [ ]:
df_final.head(10)

In [ ]:
len(df_usefulcols) / len(df_pivot)

In [ ]:
index_cols = ["year", "state", "statistic_category"]

duplicates = raw_df[raw_df.duplicated(subset=index_cols, keep=False)]
#print(duplicates.sort_values(index_cols))

In [ ]:
print(duplicates[["description", "class", "statistic_category"]].drop_duplicates())